# Single CW Sampler — Simulated Annealing + Adaptive Empirical Covariance

This notebook demonstrates the single continuous wave (CW) sampler for phase-connected pulsar timing array analyses.
It uses a simplified toy likelihood (white noise only, no red noise or GWB) to focus on the sampling method itself.

## The Problem
We sample 8 CW source parameters (sky location, inclination, chirp mass, frequency, strain, phase, polarisation)
plus one distance per pulsar. Pulsar distances create a **multimodal** likelihood — the mode spacing $\Delta L$
is set by the GW wavelength and sky geometry. Starting 3–5 sigma from truth means the chain must cross a
vast low-likelihood desert before reaching the posterior peak.

## The Solution: Simulated Annealing
**Annealing** is borrowed from metallurgy: heat metal and cool it slowly so atoms settle into a low-energy
crystal rather than freezing in a disordered state. In MCMC, we temporarily *flatten* the posterior by
dividing the log-acceptance ratio by a temperature $T > 1$:
$$\log \alpha = \frac{\log p(\theta') - \log p(\theta)}{T}$$
At high $T$ almost every proposal is accepted — the chain wanders freely. As $T$ cools toward 1 the
criterion tightens and the chain settles into the true posterior. This lets the sampler **find** the
posterior from a cold start rather than getting stuck.

## Three Phases
1. **Anneal** (`n_anneal` steps, $T$: `T_start` → 1): Fisher eigenmodes scaled by $\sqrt{T}$, per-mode
   scale adaptation targeting 35% acceptance. Distances proposed from EM prior + grid scan.
2. **Adapt** (`n_adapt` steps, $T=1$): Build empirical covariance from late-annealing samples, switch
   eigenmode proposals to empirical eigenvectors.
3. **Production** (`n_prod` steps, $T=1$): Sample the true posterior.

## Move Types
| Move | Fraction | Description |
|------|----------|-------------|
| **CW Eigenmode** | 50% | Step along one Fisher/empirical eigenmode; Newton-snap all distances |
| **Distance prior draw** | 30% | Draw distance from EM prior, grid-scan for best fringe |
| **CW Joint** | 20% | Full 8D proposal from empirical Cholesky factor |

## Setup and Imports

In [ ]:
import os
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["XLA_PYTHON_CLIENT_ALLOCATOR"] = "platform"
os.environ["JAX_DISABLE_MMAP_CACHE"] = "1"
os.environ["XLA_FLAGS"] = "--xla_gpu_autotune_level=2"

import glob, time
import numpy as np
import matplotlib.pyplot as plt

import jax
jax.config.update('jax_enable_x64', True)
jax.clear_caches()
import jax.numpy as jnp

import discovery as ds
from enterprise_extensions import load_feathers
from discovery.deterministic import make_phase_connected_binary
from discovery import const as disco_const
from discovery.deterministic import fpcmu_fast

print('Imports OK')

## Configurable Parameters

Change these to adjust the run. The sampler schedule mirrors the working multi-CW annealing notebook.

In [ ]:
# =============================================================================
# CONFIGURABLE PARAMETERS — CHANGE THESE
# =============================================================================
Npulsars = 5        # Number of pulsars to include (max 116 available)
log10_h  = -12.0    # log10(strain amplitude). -12 = high SNR, -13 = moderate

# Sampler schedule: 3 phases
n_anneal = 15000    # Phase 1: annealing steps (T: T_start -> 1)
n_adapt  = 5000     # Phase 2: adaptive covariance burn-in at T=1
n_prod   = 5000     # Phase 3: production sampling at T=1
T_start  = 5000.0   # Starting temperature (higher = broader initial exploration)
T_end    = 1.0      # Final temperature (1.0 = sample the true posterior)

# Derived constants
Ndim       = 8 + Npulsars         # 8 CW params + Npulsars distances
sigma_toa  = 1e-6                  # White noise level [seconds] applied to all pulsars
KPC_OVER_C = disco_const.kpc / disco_const.c  # kpc -> light-seconds conversion

print(f"Configuration: N_CW=1, Npulsars={Npulsars}, h=10^{log10_h}")
print(f"Dimensions: 8 CW params + {Npulsars} distances = {Ndim} total")
print(f"Sampler: {n_anneal} anneal + {n_adapt} adapt + {n_prod} prod = {n_anneal+n_adapt+n_prod} total")

## Load Pulsars and EM Distance Priors

Load pulsar data products and extract the electromagnetic (EM) distance priors (mean and uncertainty)
for each pulsar. These priors constrain pulsar distances in the likelihood and are used to draw
distance proposals.

In [ ]:
feather_dir = "../data_products/"

disco_psrs = [ds.Pulsar.read_feather(f) for f in sorted(glob.glob(feather_dir + "*.feather"))][:Npulsars]
for psr in disco_psrs:
    # Override TOA errors with uniform white noise level
    psr.toaerrs = np.full_like(psr.toas, sigma_toa, dtype=np.float32)
print(f"Loaded {len(disco_psrs)} pulsars: {[p.name for p in disco_psrs]}")

# Load EM distance priors from enterprise pulsar objects
psrs_ent   = load_feathers.load_feathers_from_folder(feather_dir)
ent_by_name = {p.name: p for p in psrs_ent}

dist_mu, dist_sig = [], []
for psr in disco_psrs:
    ep  = ent_by_name[psr.name]
    mu  = float(ep.pdist[0])                                  # EM distance mean [kpc]
    sig = float(ep.pdist[1]) if len(ep.pdist) > 1 else 0.5   # EM distance sigma [kpc]
    if (not np.isfinite(sig)) or sig <= 0:
        sig = 0.5  # fallback if missing
    dist_mu.append(mu)
    dist_sig.append(sig)

# JAX arrays for use inside logp
dist_mu_jnp = jnp.array(dist_mu, dtype=jnp.float64)
sd_jnp      = jnp.array(dist_sig, dtype=jnp.float64)

# Numpy arrays for proposals
mu_arr = np.array(dist_mu)
sd_arr = np.array(dist_sig)

# Pre-extract TOAs and sky positions for each pulsar
psr_toas_list = [np.asarray(psr.toas, dtype=np.float64) for psr in disco_psrs]
psr_pos_list  = [psr.pos for psr in disco_psrs]              # unit 3-vectors
psr_positions = jnp.array([psr.pos for psr in disco_psrs])   # (Npulsars, 3) for vmap

print(f"dist_mu:  {[f'{m:.3f}' for m in dist_mu]}")
print(f"dist_sig: {[f'{s:.4f}' for s in dist_sig]}")

## CW Injection and Log-Posterior

Inject a single CW source into the data. The model uses `make_phase_connected_binary` from `discovery`,
which computes the deterministic CW delay for each pulsar including both the Earth term and the pulsar term.

The log-posterior is:
$$\log p(\theta | d) = -\frac{1}{2\sigma_{\rm TOA}^2} \sum_j (d_j - m_j(\theta))^2
  - \frac{1}{2} \sum_j \frac{(L_j - \mu_j^{\rm EM})^2}{\sigma_j^{{\rm EM}\,2}}$$

where $\theta$ = (8 CW params, $N_{\rm psr}$ distances), and the second term is the Gaussian EM distance prior.
The prior on distances prevents the chain from wandering to unphysical distance values far from any
measured pulsar distance.

In [ ]:
CW_PARAM_NAMES = ['cos_gwtheta', 'gwphi', 'cos_inc', 'log10_mc', 'log10_fgw', 'log10_h', 'phase0', 'psi']

# ----- Injection parameters -----
INJ = {
    "cos_gwtheta": 0.3,  "gwphi":    2.5,  "cos_inc":  -0.2,
    "log10_mc":    9.0,  "log10_fgw": -8.0, "log10_h": log10_h,
    "phase0":      1.0,  "psi":       0.7,
}

cw_func = make_phase_connected_binary(pulsarterm=True)

# ----- True pulsar distances (offset from EM prior mean by 0.3 sigma) -----
# Small offset so the chain starts near-but-not-at the right distance mode.
DIST_OFFSET_SIGMA = 0.3
true_dists = {psr.name: float(dist_mu[i]) + DIST_OFFSET_SIGMA * float(dist_sig[i])
              for i, psr in enumerate(disco_psrs)}

# ----- Generate data: CW signal + (implicit) noise -----
data_list = []
for psr in disco_psrs:
    toas_i  = np.asarray(psr.toas, dtype=np.float64)
    delay_i = cw_func(toas_i, psr.pos, p_dist=true_dists[psr.name], **INJ)
    data_list.append(np.array(delay_i, dtype=np.float64))

# SNR
snr2 = sum(
    np.sum(np.array(cw_func(psr_toas_list[i], psr.pos, p_dist=true_dists[psr.name], **INJ))**2) / sigma_toa**2
    for i, psr in enumerate(disco_psrs)
)
print(f"Injected source: SNR={np.sqrt(snr2):.1f}, fgw=10^{INJ['log10_fgw']:.2f}, h=10^{log10_h}")

# ----- Build truth vector -----
truth_cw   = [INJ[k] for k in CW_PARAM_NAMES]
truth_dist = [true_dists[psr.name] for psr in disco_psrs]
truth      = np.array(truth_cw + truth_dist)

# JAX data arrays
data_jnp = [jnp.array(d) for d in data_list]

# ----- Parameter bounds -----
CW_BOUNDS_LO = jnp.array([-1.0, 0.0,        -1.0, 7.0, -9.0, -18.0, 0.0,        0.0])
CW_BOUNDS_HI = jnp.array([ 1.0, 2*jnp.pi,   1.0, 10.0, -7.0, -11.0, 2*jnp.pi, jnp.pi])
PARAM_LO = np.array([float(CW_BOUNDS_LO[i]) for i in range(8)] + [1e-6]*Npulsars)
PARAM_HI = np.array([float(CW_BOUNDS_HI[i]) for i in range(8)] + [30.0]*Npulsars)

# ----- Log-posterior -----
@jax.jit
def logp(x):
    """Log-posterior for one CW source + Npulsars distances.
    
    Parameter vector layout:
      x[0:8]          = CW params (cos_gwtheta, gwphi, cos_inc, log10_mc, log10_fgw,
                                    log10_h, phase0, psi)
      x[8:8+Npulsars] = pulsar distances [kpc]
    """
    p_dists = x[8:8 + Npulsars]

    # --- Bounds check: return -1e30 if any parameter is outside prior support ---
    in_bounds = (jnp.all(p_dists > 1e-6)
                 & jnp.all(x[:8] >= CW_BOUNDS_LO)
                 & jnp.all(x[:8] <= CW_BOUNDS_HI))

    # --- Gaussian likelihood: chi-squared residuals ---
    ll = 0.0
    for p_idx in range(Npulsars):
        model = cw_func(
            psr_toas_list[p_idx], psr_pos_list[p_idx],
            cos_gwtheta=x[0], gwphi=x[1], cos_inc=x[2],
            log10_mc=x[3], log10_fgw=x[4], log10_h=x[5],
            phase0=x[6], psi=x[7], p_dist=p_dists[p_idx], p_phase=None
        )
        resid = data_jnp[p_idx] - model
        ll   -= 0.5 * jnp.sum(resid**2) / sigma_toa**2

    # --- Gaussian EM distance prior ---
    log_prior = -0.5 * jnp.sum(jnp.square((p_dists - dist_mu_jnp) / sd_jnp))

    return jnp.where(in_bounds, ll + log_prior, -1e30)

# Compile and verify
lp_truth = float(logp(jnp.array(truth)))
print(f"logp(truth) = {lp_truth:.4f}")

# ----- Distance fringe spacing -----
# The pulsar term creates a periodic fringe pattern in distance with spacing:
#   delta_L = 1 / (f_gw * (kpc/c) * |1 - cos mu|)
# This sets the scale of distance proposals — steps must be << delta_L to stay on-mode.
@jax.jit
def compute_delta_L(cos_gwtheta, gwphi, log10_fgw):
    """Distance fringe spacing [kpc] for each pulsar."""
    gwtheta = jnp.arccos(cos_gwtheta)
    f_gw    = 10.0 ** log10_fgw
    _, _, cos_mu = jax.vmap(lambda pos: fpcmu_fast(pos, gwtheta, gwphi))(psr_positions)
    denom = jnp.maximum(jnp.abs(1.0 - cos_mu), 1e-4)
    return 1.0 / (f_gw * KPC_OVER_C * denom)

dL = np.array(compute_delta_L(truth[0], truth[1], truth[4]))
print("Distance fringe spacings and modes per sigma:")
for i, psr in enumerate(disco_psrs):
    print(f"  {psr.name}: d_true={truth[8+i]:.4f} kpc, dL={dL[i]:.6f} kpc, modes/sig={sd_arr[i]/dL[i]:.0f}")

## Fisher Information and Starting Point

Compute the Hessian of the log-posterior at truth to get:
- **8×8 CW block** → eigendecomposition gives Fisher eigenmode directions and widths for initial proposals
- **Distance diagonals** → per-pulsar curvatures for Newton distance snapping

The Fisher eigenmodes are used as initial proposals during annealing. They are replaced by the
empirical chain covariance at the adapt phase, which better captures the true posterior geometry.

The starting point is offset 3–5 Fisher sigmas from truth in each CW dimension, and 3–5 EM sigmas
in each distance — this tests that the annealing can actually find the posterior from a cold start.

In [ ]:
# JIT-compiled gradient and batch logp (used for Newton snapping and distance grid scans)
grad_logp = jax.jit(jax.grad(logp))
batch_logp = jax.jit(jax.vmap(logp))

# ----- Hessian at truth -----
print("Computing Hessian at truth...")
t0     = time.time()
H_full = np.array(jax.hessian(logp)(jnp.array(truth)))
print(f"Hessian computed in {time.time()-t0:.1f}s")

# ----- CW block: eigendecompose to get Fisher proposal directions -----
H_cw   = H_full[:8, :8]                          # 8x8 CW-CW block
eig_cw, evec_cw = np.linalg.eigh(-H_cw)          # eigenvalues of Fisher info (positive)
eig_cw_c = np.maximum(eig_cw, 1e-12 * eig_cw.max())  # clip tiny eigenvalues

# Fisher covariance = inverse of Fisher information
cov_fisher   = evec_cw @ np.diag(1.0 / eig_cw_c) @ evec_cw.T
cov_fisher   = 0.5 * (cov_fisher + cov_fisher.T)  # enforce symmetry

# Eigendecompose the covariance to get proposal directions and widths
eig_vals_cov, eig_vecs_cov = np.linalg.eigh(cov_fisher)
eig_sigs_fisher = 2.38 * np.sqrt(np.maximum(eig_vals_cov, 1e-30))  # optimal 1D MH scale

# Per-parameter Fisher sigma (for building the starting point)
fisher_sig_cw = np.sqrt(np.diag(cov_fisher))

# ----- Distance block: diagonal curvatures for Newton snapping -----
# The Hessian diagonal for each distance tells us the curvature of the likelihood peak,
# which we use to Newton-step each distance toward its local maximum.
H_dist_diag = np.array([H_full[8+j, 8+j] for j in range(Npulsars)])

print(f"Fisher eigenmode widths: min={eig_sigs_fisher.min():.2e}, max={eig_sigs_fisher.max():.2e}")

# Pre-compile batch_logp
print("Pre-compiling batch_logp...")
_ = batch_logp(jnp.tile(jnp.array(truth), (40, 1)))
print("Done.")

# ----- Build starting point: offset 3-5 sigma from truth -----
# This tests that the annealing can actually find the posterior from a cold start.
# Starting from truth would give an unrealistically optimistic picture of sampler performance.
rng = np.random.default_rng(99)
x0  = truth.copy()

# Offset CW parameters by 3-5 Fisher sigmas in a random direction
for i in range(8):
    offset_sigma = rng.uniform(3.0, 5.0) * rng.choice([-1, 1])
    proposed     = x0[i] + offset_sigma * fisher_sig_cw[i]
    lo = float(CW_BOUNDS_LO[i]) + 1e-4
    hi = float(CW_BOUNDS_HI[i]) - 1e-4
    x0[i] = np.clip(proposed, lo, hi)

# Offset distances by 3-5 EM sigmas
for j in range(Npulsars):
    offset     = rng.uniform(3.0, 5.0) * rng.choice([-1, 1]) * sd_arr[j]
    x0[8+j]    = max(truth[8+j] + offset, 0.01)

lp_start = float(logp(jnp.array(x0)))
print(f"logp(start) = {lp_start:.2f}")
print(f"logp(truth) = {lp_truth:.4f}")
print(f"Gap: {lp_truth - lp_start:.0f} nats")

## The Sampler

Three helper functions used inside the loop:

- **`snap_distances`**: After a CW parameter update, use Newton iterations (gradient / diagonal curvature)
  to shift each pulsar distance toward its nearest likelihood peak. Without this, any CW move that
  shifts the optimal distance would almost always be rejected.
- **`in_bounds`**: Check that the proposed point is within all prior bounds.

The main loop runs `n_anneal + n_adapt + n_prod` steps total, with temperature managed automatically.

In [ ]:
def snap_distances(x_prop, n_newton=3):
    """Newton-snap all pulsar distances toward their local likelihood peak.
    
    After updating CW parameters, the optimal distance shifts. Rather than waiting
    for the MCMC to slowly find it, we do a few Newton steps:
        d_new = d_old - gradient / curvature
    This dramatically improves mixing — otherwise CW proposals are almost always rejected.
    """
    for _ in range(n_newton):
        g = np.array(grad_logp(jnp.array(x_prop)))
        for j in range(Npulsars):
            if H_dist_diag[j] < -1e-6:  # only snap if curvature is well-defined
                x_prop[8+j] = max(x_prop[8+j] - g[8+j] / H_dist_diag[j], 1e-6)
    return x_prop


def in_bounds(x_prop):
    """Check all parameters are within prior bounds."""
    if np.any(x_prop[8:] <= 1e-6):
        return False
    if np.any(x_prop[:8] < PARAM_LO[:8]) or np.any(x_prop[:8] > PARAM_HI[:8]):
        return False
    return True


# ----- Geometric cooling schedule -----
# T(step) = T_start * cool_rate^step, cooling from T_start to T_end over n_anneal steps
cool_rate = (T_end / T_start) ** (1.0 / n_anneal)
n_total   = n_anneal + n_adapt + n_prod

print(f"Annealing: T {T_start} -> {T_end} over {n_anneal} steps (cool_rate={cool_rate:.6f})")
print(f"Adapt:     {n_adapt} steps at T=1 to build empirical covariance")
print(f"Prod:      {n_prod} steps")

# ----- Storage for the full chain -----
all_chain = np.zeros((n_total, Ndim))  # parameter values at each step
all_lps   = np.zeros(n_total)          # log-posterior at each step
all_temps = np.zeros(n_total)          # temperature at each step

# ----- Sampler state -----
x  = x0.copy()                         # current position
lp = float(logp(jnp.array(x)))         # current log-posterior
T  = T_start                           # current temperature

# ----- Empirical covariance accumulator -----
# During the second half of annealing, save samples to build empirical covariance.
# The empirical cov captures the true posterior geometry better than the Fisher matrix.
emp_samples = []
emp_cov     = None       # will hold {'L': Cholesky, 'eig_sigs': widths, 'vecs': eigenvectors}
use_emp_cov = False      # switched to True after building empirical cov

# ----- Per-eigenmode scale adaptation (annealing only) -----
# Each eigenmode has a log-scale factor adapted via Robbins-Monro to target 35% acceptance.
scale_log = np.zeros(8)

# ----- Acceptance tracking -----
acc_counts = {'eigen': 0, 'dist': 0, 'joint': 0}
tot_counts = {'eigen': 0, 'dist': 0, 'joint': 0}

t0 = time.time()

for step in range(n_total):
    # ----- Determine current phase and temperature -----
    if step < n_anneal:
        phase = 'anneal'
        T     = T_start * (cool_rate ** step)  # geometric cooling
    else:
        phase = 'adapt' if step < n_anneal + n_adapt else 'prod'
        T     = 1.0

    # ----- Phase transition: build empirical covariance -----
    # At the end of annealing, build the empirical covariance from the late-annealing samples.
    # These samples were taken when T was close to 1, so they reflect the real posterior geometry.
    if step == n_anneal and len(emp_samples) > 8 * 2:
        emp_arr = np.array(emp_samples)
        last_n  = max(8 * 3, len(emp_arr) // 3)   # use last 1/3 of annealing samples
        emp_arr = emp_arr[-last_n:, :8]             # CW params only

        if emp_arr.shape[0] > 8:
            raw_cov = np.cov(emp_arr.T)
            raw_cov = 0.5 * (raw_cov + raw_cov.T)  # enforce symmetry

            # Regularise: clip tiny eigenvalues to avoid singular covariance
            eig_e, vec_e = np.linalg.eigh(raw_cov)
            eig_e        = np.maximum(eig_e, 1e-12 * eig_e.max())
            emp_cov_mat  = vec_e @ np.diag(eig_e) @ vec_e.T

            # Optimal MH scaling: 2.38^2/d for d-dimensional Gaussian target
            scale_nd = 2.38**2 / 8
            try:
                L_emp   = np.linalg.cholesky(scale_nd * emp_cov_mat)
                emp_cov = {
                    'L':        L_emp,                        # Cholesky factor for joint proposals
                    'eig_sigs': 2.38 * np.sqrt(eig_e),        # per-eigenmode widths
                    'vecs':     vec_e,                         # eigenvectors
                }
                use_emp_cov = True
                print(f"  [step {step}] Switched to empirical covariance "
                      f"(built from {emp_arr.shape[0]} samples)")
            except np.linalg.LinAlgError:
                print(f"  [step {step}] Empirical Cholesky failed, keeping Fisher cov")

    # ----- Choose and execute a proposal -----
    r = rng.random()

    if r < 0.50:
        # === EIGENMODE PROPOSAL (50% of steps) ===
        # Step along one eigenvector of the CW covariance.
        # During annealing: Fisher eigenmodes scaled by sqrt(T) — wide when hot, narrow when cold.
        # After transition: empirical eigenmodes from the chain itself.
        if use_emp_cov:
            mode_idx = rng.integers(8)
            z        = rng.standard_normal()
            sig      = emp_cov['eig_sigs'][mode_idx]
            x_prop   = x.copy()
            x_prop[:8] += z * sig * emp_cov['vecs'][:, mode_idx]
        else:
            mode_idx   = rng.integers(8)
            z          = rng.standard_normal()
            # Scale by sqrt(T): proposals are sqrt(T) times wider at temperature T
            scaled_sig = eig_sigs_fisher[mode_idx] * np.sqrt(T) * np.exp(scale_log[mode_idx])
            x_prop     = x.copy()
            x_prop[:8] += z * scaled_sig * eig_vecs_cov[:, mode_idx]

        # Newton-snap distances to their local optimum after CW param change
        x_prop = snap_distances(x_prop)

        if in_bounds(x_prop):
            lp_prop  = float(logp(jnp.array(x_prop)))
            # Tempered Metropolis-Hastings: dividing by T flattens the acceptance criterion
            log_alpha = (lp_prop - lp) / T
            accepted  = np.log(rng.random() + 1e-300) < log_alpha
            if accepted:
                x = x_prop; lp = lp_prop
            acc_counts['eigen'] += int(accepted)
        tot_counts['eigen'] += 1

        # Per-eigenmode scale adaptation (annealing only)
        # Robbins-Monro: nudge log-scale up if accepted, down if rejected, targeting 35%
        if phase == 'anneal':
            gamma = 1.0 / (step + 100)  # decaying step size ensures convergence
            scale_log[mode_idx] += gamma * (float(x is x_prop) - 0.35)
            scale_log[mode_idx]  = np.clip(scale_log[mode_idx], -5.0, 10.0)

    elif r < 0.80:
        # === DISTANCE PROPOSAL (30% of steps) ===
        # Draw a new distance from the EM prior, then scan a grid around it to find the
        # best distance fringe. This handles the highly multimodal distance likelihood
        # (many narrow fringes spaced by delta_L ~ 0.0006 kpc over a prior width of ~0.1 kpc).
        pi    = rng.integers(Npulsars)
        d_prop = rng.normal(mu_arr[pi], sd_arr[pi] * max(1.0, np.sqrt(T)))
        dL_j  = float(dL[pi])  # fringe spacing for this pulsar

        if d_prop > dL_j:  # must be positive and > one fringe width
            x_snap         = x.copy()
            x_snap[8+pi]   = d_prop

            # Scan a grid of +/-0.6 fringe widths to find the best fringe
            d_lo    = max(d_prop - 0.6 * dL_j, 1e-6)
            d_hi    = d_prop + 0.6 * dL_j
            d_cands = np.linspace(d_lo, d_hi, 30)
            x_batch = np.tile(x_snap, (30, 1))
            x_batch[:, 8+pi] = d_cands
            lps_scan = np.array(batch_logp(jnp.array(x_batch)))

            x_prop         = x.copy()
            x_prop[8+pi]   = float(d_cands[np.argmax(lps_scan)])
            lp_prop        = float(logp(jnp.array(x_prop)))

            log_alpha = (lp_prop - lp) / T
            if np.log(rng.random() + 1e-300) < log_alpha:
                x = x_prop; lp = lp_prop
                acc_counts['dist'] += 1
        tot_counts['dist'] += 1

    else:
        # === JOINT CW PROPOSAL (20% of steps) ===
        # Full 8D Gaussian proposal using the Cholesky factor of the covariance.
        # Enables correlated moves across all CW params simultaneously.
        if use_emp_cov:
            z      = rng.standard_normal(8)
            x_prop = x.copy()
            x_prop[:8] += emp_cov['L'] @ z
        else:
            scale_nd = 2.38**2 / 8
            L_fish   = np.linalg.cholesky(scale_nd * T * cov_fisher)
            z        = rng.standard_normal(8)
            x_prop   = x.copy()
            x_prop[:8] += L_fish @ z

        if in_bounds(x_prop):
            lp_prop   = float(logp(jnp.array(x_prop)))
            log_alpha = (lp_prop - lp) / T
            if np.log(rng.random() + 1e-300) < log_alpha:
                x = x_prop; lp = lp_prop
                acc_counts['joint'] += 1
        tot_counts['joint'] += 1

    # ----- Record state -----
    all_chain[step] = x
    all_lps[step]   = lp
    all_temps[step] = T

    # Accumulate samples for empirical covariance (last half of annealing)
    if phase == 'anneal' and step > n_anneal // 2:
        emp_samples.append(x.copy())

    # Progress logging every 2000 steps
    if step % 2000 == 0:
        elapsed = time.time() - t0
        print(f"  step {step:5d}/{n_total} [{phase:6s}] T={T:7.2f} | logp={lp:10.2f} | "
              f"eigen={acc_counts['eigen']}/{tot_counts['eigen']} "
              f"dist={acc_counts['dist']}/{tot_counts['dist']} "
              f"joint={acc_counts['joint']}/{tot_counts['joint']} | {elapsed:.0f}s")

# ----- Summary -----
dt = time.time() - t0
print(f"\nDone in {dt:.1f}s ({n_total/dt:.0f} it/s)")
for k in acc_counts:
    ar = acc_counts[k] / max(tot_counts[k], 1)
    print(f"  {k:10s}: {acc_counts[k]:5d}/{tot_counts[k]:5d} = {ar:.3f}")

prod_chain = all_chain[n_anneal + n_adapt:]
prod_lps   = all_lps[n_anneal + n_adapt:]

print(f"\nlogp truth = {lp_truth:.4f}")
print(f"logp prod:  mean={np.mean(prod_lps):.2f}, std={np.std(prod_lps):.2f}, max={np.max(prod_lps):.2f}")

print("\nCW parameter recovery (production chain):")
for pidx, pname in enumerate(CW_PARAM_NAMES):
    med  = np.median(prod_chain[:, pidx])
    std  = np.std(prod_chain[:, pidx])
    bias = med - truth[pidx]
    print(f"  {pname:15s}: truth={truth[pidx]:+.4f}, median={med:+.4f}, std={std:.2e}, bias={bias:+.4f}")

print("\nDistance recovery (production chain):")
for j in range(Npulsars):
    med       = np.median(prod_chain[:, 8+j])
    err_modes = abs(med - truth[8+j]) / dL[j]
    print(f"  {disco_psrs[j].name}: med={med:.4f}, truth={truth[8+j]:.4f}, err={err_modes:.0f} fringe modes")

## Plots

Four rows:
- **Row 0**: logp trace, temperature schedule, log10_fgw trace
- **Row 1**: cos_gwtheta, log10_h, cos_inc (key CW params)
- **Row 2**: Pulsar distance traces (B1855+09, B1937+21, B1953+29)
- **Row 3**: Pulsar distance traces (J0023+0923, J0030+0451), log10_h posterior histogram

Blue dashed line = end of annealing. Orange dashed line = end of adapt. Red dashed = truth.
Orange dot = starting point.

In [ ]:
ann_end   = n_anneal
adap_end  = n_anneal + n_adapt
steps_arr = np.arange(n_total)

fig, axes = plt.subplots(4, 3, figsize=(18, 16))
fig.suptitle(
    f'N_CW=1, h=10^{log10_h}: Annealing (T:{T_start:.0f}->1) + Adaptive Cov + Production\n'
    f'Start: dlogp={lp_start-lp_truth:.0f} from truth | {n_anneal}+{n_adapt}+{n_prod} steps',
    fontsize=13)

colour = '#1a3a5c'

def add_phase_lines(ax):
    ax.axvline(ann_end,  color='blue',   ls=':', lw=1.5, alpha=0.7)
    ax.axvline(adap_end, color='orange', ls=':', lw=1.5, alpha=0.7)

# --- Row 0, Col 0: logp trace ---
ax = axes[0, 0]
ax.plot(steps_arr, all_lps, color='#333', lw=0.3, alpha=0.8)
ax.axhline(lp_truth, color='r', ls='--', lw=1, label=f'truth ({lp_truth:.2f})')
add_phase_lines(ax)
ax.set_xlabel('step'); ax.set_ylabel('logp')
ax.set_title('Log-posterior trace'); ax.legend(fontsize=7)

# --- Row 0, Col 1: temperature schedule ---
ax = axes[0, 1]
ax.semilogy(steps_arr[:n_anneal], all_temps[:n_anneal], color='#b5442d', lw=0.6)
ax.axhline(1.0, color='gray', ls='--', lw=1)
ax.set_xlabel('step'); ax.set_ylabel('Temperature T')
ax.set_title('Temperature schedule (annealing phase)')

# --- Row 0, Col 2: log10_fgw ---
ax = axes[0, 2]
ax.plot(steps_arr, all_chain[:, 4], color=colour, lw=0.3, alpha=0.8)
ax.axhline(truth[4], color='r', ls='--', lw=1)
ax.plot(0, x0[4], 'o', color=colour, ms=5, zorder=5)
add_phase_lines(ax)
ax.set_title('log10_fgw'); ax.set_xlabel('step')

# --- Row 1: key CW parameters ---
for pidx, (param_col, param_name) in enumerate([(0, 'cos_gwtheta'), (5, 'log10_h'), (2, 'cos_inc')]):
    ax = axes[1, pidx]
    ax.plot(steps_arr, all_chain[:, param_col], color=colour, lw=0.3, alpha=0.8)
    ax.axhline(truth[param_col], color='r', ls='--', lw=1)
    ax.plot(0, x0[param_col], 'o', color=colour, ms=5, zorder=5)
    add_phase_lines(ax)
    ax.set_title(param_name); ax.set_xlabel('step')

# --- Rows 2-3: distance traces and log10_h posterior ---
dist_slots = [(2, 0), (2, 1), (2, 2), (3, 0), (3, 1)]  # (row, col) for each pulsar
for j in range(min(Npulsars, 5)):
    row, col = dist_slots[j]
    ax = axes[row, col]
    ax.plot(steps_arr, all_chain[:, 8+j], color=colour, lw=0.3, alpha=0.8)
    ax.axhline(truth[8+j], color='r', ls='--', lw=1, label='truth')
    ax.plot(0, x0[8+j], 'o', color='orange', ms=5, zorder=5, label='start')
    add_phase_lines(ax)
    ax.set_title(f'{disco_psrs[j].name} dist'); ax.set_xlabel('step')
    ax.legend(fontsize=7)

# --- Row 3, Col 2: log10_h posterior (production only) ---
ax = axes[3, 2]
ax.hist(prod_chain[:, 5], bins=50, color=colour, alpha=0.7)
ax.axvline(truth[5], color='r', ls='--', lw=1.5, label=f'truth={truth[5]}')
ax.set_title('log10_h posterior (production)'); ax.set_xlabel('log10_h'); ax.legend(fontsize=7)

plt.tight_layout()
plt.show()